# Same-image GQA diagnostic pilot

Run after the full MMStar process finishes and its ZIP is downloaded. Enable Internet and T4 ×2. Native ApET code and dependencies stay unchanged. Default: 10 images, four questions per image, four configurations, one model load. See `research/GQA_DIAGNOSTIC_PILOT.md`.

## 1. Update code and choose paths

In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True)
subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path(f"/kaggle/working/original-gqa-pilot-{stamp}")
run_dir.mkdir()
original_python = repo / "environments/original/.venv/bin/python"
data_dir = Path("/kaggle/working/gqa-pilot-data-v1-10x4-s590")
PREPARED_MANIFEST = None  # Optional: "/kaggle/input/your-subset/manifest.json"
data_manifest = Path(PREPARED_MANIFEST) if PREPARED_MANIFEST else data_dir / "manifest.json"
report_path = run_dir / "original-gqa-pilot.json"
checkpoint_args = []  # Optional original-format local model/vision paths, as in MMStar.


## 2. Check runtime/logging and prepare data

Only two pinned Test-Dev files (~68 MB) download. Native logging tests use synthetic tensors without checkpoints. Existing prepared data can be reused.

In [ ]:
import shutil

subprocess.run(
    ["bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
     "--output", str(run_dir / "original-runtime.json")],
    cwd=repo, check=True,
)
subprocess.run(
    [str(original_python), "-I", "research/tests/test_original_diagnostics_native.py"],
    cwd=repo, check=True,
)
if not data_manifest.is_file():
    if PREPARED_MANIFEST:
        raise FileNotFoundError(f"Attached manifest not found: {data_manifest}")
    subprocess.run(
        [str(original_python), "-I", "scripts/prepare_gqa_pilot.py", "--output-dir", str(data_dir),
         "--image-count", "10", "--questions-per-image", "4", "--seed", "590", "--no-zip"],
        cwd=repo, check=True,
    )
data = json.loads(data_manifest.read_text())
print("Images:", data["selection"]["image_count"], "Questions:", len(data["questions"]))
print("Question types:", data["selection"]["type_counts"])

reference = json.loads((repo / "research/provenance/original_llava_reference.json").read_text())
snapshot_files = [*reference["native_source_sha256"],
    "scripts/check_original_env.py", "scripts/smoke_original_apet.py", "scripts/eval_mmstar.py",
    "scripts/eval_original_mmstar.py", "scripts/eval_original_gqa.py", "scripts/prepare_gqa_pilot.py",
    "scripts/gqa_scoring.py", "scripts/original_diagnostics.py", "scripts/convert_gqa_for_eval.py",
    "environments/original/pyproject.toml", "environments/original/uv.lock",
    "environments/original/.python-version", "bootstrap/setup_original_linux.sh",
    "research/provenance/original_llava_reference.json", "research/IMPLEMENTATION_SCOPE.md",
]
for name in snapshot_files:
    destination = run_dir / "source" / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(repo / name, destination)


## 3. Run 160 predictions plus controls

Read-only observations save reconstruction errors, basis/selected indices, and merge assignments. Logging off/on controls must pass. Answer-contribution interventions are not included yet. Diagnostic timing includes observer overhead.

In [ ]:
subprocess.run(
    [str(original_python), "-I", "scripts/eval_original_gqa.py", "--data-manifest", str(data_manifest),
     "--configurations", "baseline", "input-only", "decoder-only", "combined",
     "--max-new-tokens", "16", "--download-budget-gib", "20", *checkpoint_args,
     "--output", str(report_path)],
    cwd=repo, check=True,
)


## 4. Inspect and download evidence

This cell can bundle partial evidence after a failure. Source snapshots, selected images, questions and raw diagnostic records are included; model caches are excluded.

In [ ]:
import zipfile
from IPython.display import FileLink, display

if report_path.is_file():
    result = json.loads(report_path.read_text())
    print("Status:", result["status"], "Progress:", result.get("progress"))
    print("Observer parity:", result.get("observer_parity"))
    if result.get("error"):
        print("Error:", result["error"])
    for name, metrics in result.get("summaries", {}).items():
        print(name, "correct:", metrics["correct"], "/", metrics["evaluated"],
              "accuracy:", metrics["accuracy"],
              "APR:", result.get("comparisons", {}).get(name, {}).get("apr"))
bundle = run_dir.with_suffix(".zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.rglob("*")):
        if path.is_file():
            archive.write(path, str(path.relative_to(run_dir)))
print(f"Download before ending the session: {bundle}")
display(FileLink(str(bundle)))
